# Linear Regression: Closed-Form Normal Equations and Statistical Inference
Accompanies **Unit 3.1: Linear Regression from First Principles**.

The Ordinary Least Squares (OLS) estimator minimizes the sum of squared residuals:
$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In this notebook, we:
1. Derive and compute the closed-form normal equations from raw NumPy arrays.
2. Verify exact numerical equivalence against `scikit-learn`.
3. Perform formal statistical inference with `statsmodels`, examining standard errors, $t$-statistics, $p$-values, and confidence intervals.
4. Evaluate predictive performance using Mean Squared Error (MSE) and $R^2$.

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

data_path = os.path.abspath('../../data/loan_data_train.csv')
df = pd.read_csv(data_path)

# Prepare clean subset of continuous features
df_clean = df[['Amount.Requested', 'Monthly.Income', 'Open.CREDIT.Lines', 'Interest.Rate']].copy()
df_clean['Interest.Rate'] = pd.to_numeric(df_clean['Interest.Rate'].astype(str).str.replace('%', '', regex=False), errors='coerce')
df_clean = df_clean.dropna()

X = df_clean[['Amount.Requested', 'Monthly.Income', 'Open.CREDIT.Lines']].values
y = df_clean['Interest.Rate'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Computing the Normal Equations in Pure NumPy
To estimate the intercept $w_0$, we prepend a column of ones to design matrix $\mathbf{X}$:

In [ ]:
# Prepend column of ones
X_design = np.column_stack([np.ones(len(X_train)), X_train])

# Normal equations: (X^T X)^{-1} X^T y
XT_X = X_design.T @ X_design
XT_y = X_design.T @ y_train
w_numpy = np.linalg.inv(XT_X) @ XT_y

print("NumPy Closed-Form Weights [w0, w1, w2, w3]:")
print(w_numpy)

### Verification with Scikit-Learn

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
w_sklearn = np.array([lr.intercept_] + list(lr.coef_))

max_discrepancy = np.max(np.abs(w_numpy - w_sklearn))
print(f"Maximum discrepancy between pure NumPy and Scikit-Learn: {max_discrepancy:.2e}")
assert max_discrepancy < 1e-9, "Normal equations do not match scikit-learn!"

### Statistical Inference with Statsmodels: Standard Errors and $p$-Values
In machine learning, we predict; in statistics, we infer. 
The sampling distribution of weights is:
$$\hat{\boldsymbol{\beta}} \sim \mathcal{N}\left(\boldsymbol{\beta}, \sigma^2 (\mathbf{X}^T \mathbf{X})^{-1}\right)$$
We inspect the OLS summary table to assess the $t$-statistic and $p$-value for each coefficient.

In [ ]:
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm).fit()

print(ols_model.summary())

### Holdout Test Evaluation: MSE and $R^2$

In [ ]:
y_pred = lr.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Test Mean Squared Error (MSE):  {mse:.4f}")
print(f"Test Root Mean Squared Error:   {rmse:.4f}")
print(f"Test Coefficient of Determ (R2): {r2:.4f}")